In [21]:
import os
from dotenv import load_dotenv
import openai
from openai import OpenAI
import pandas as pd
from geopy.distance import geodesic
import folium

# Load environment variables from .env file
load_dotenv()

# Set your OpenAI API key
openai.api_key = os.getenv("OPENAI_API_KEY")

In [22]:
# Initialize OpenAI client
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [34]:
def getnames(place_name):
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"List all the name variants and language variants for the place, '{place_name}', and provide its geographic coordinates (latitude and longitude)."
            }
        ],
        model="gpt-3.5-turbo",
    )

    # Extracting the response content
    response_content = chat_completion.choices[0].message.content.strip()
    return response_content

def parse_coordinates(response):
    try:
        # Extract latitude and longitude values
        lat_str = response.split("Latitude: ")[1].split("\n")[0].strip("° N° S")
        lon_str = response.split("Longitude: ")[1].split("\n")[0].strip("° E° W")

        # Convert strings to floats
        lat = float(lat_str)
        lon = float(lon_str)

        # Adjust sign based on compass direction
        if "S" in response.split("Latitude: ")[1].split("\n")[0]:
            lat = -lat
        if "W" in response.split("Longitude: ")[1].split("\n")[0]:
            lon = -lon

        return lat, lon
    except (IndexError, ValueError):
        return None, None

def compare_results(df):
    results = []

    # Limit to the first 10 rows
    limited_df = df.head(10)

    for index, row in limited_df.iterrows():
        place_name = row["place_name"]
        true_lat = row["latitude"]
        true_lon = row["longitude"]
        category = row["category"]

        # Query OpenAI
        openai_result = getnames(place_name)
        print(f"OpenAI result for {place_name}: {openai_result}")

        # Extract coordinates from OpenAI result
        openai_lat, openai_lon = parse_coordinates(openai_result)

        # Calculate the distance between true coordinates and OpenAI coordinates
        if openai_lat is not None and openai_lon is not None:
            true_coords = (true_lat, true_lon)
            openai_coords = (openai_lat, openai_lon)
            distance = geodesic(true_coords, openai_coords).kilometers
        else:
            distance = None

        results.append({
            "place_name": place_name,
            "true_latitude": true_lat,
            "true_longitude": true_lon,
            "openai_latitude": openai_lat,
            "openai_longitude": openai_lon,
            "distance_km": distance,
            "openai_result": openai_result
        })

    results_df = pd.DataFrame(results)
    return results_df

def generate_map(results_df):
    # Create a base map
    m = folium.Map(location=[results_df["true_latitude"].mean(), results_df["true_longitude"].mean()], zoom_start=2)

    for _, row in results_df.iterrows():
        true_coords = [row["true_latitude"], row["true_longitude"]]
        openai_coords = [row["openai_latitude"], row["openai_longitude"]]

        # Add true location marker
        folium.Marker(
            location=true_coords,
            popup=f'True Location: {row["place_name"]}',
            icon=folium.Icon(color='green')
        ).add_to(m)

        # Add OpenAI location marker
        if not pd.isna(row["openai_latitude"]) and not pd.isna(row["openai_longitude"]):
            folium.Marker(
                location=openai_coords,
                popup=f'OpenAI Location: {row["place_name"]}',
                icon=folium.Icon(color='red')
            ).add_to(m)

            # Add a line between true location and OpenAI location
            folium.PolyLine(locations=[true_coords, openai_coords], color='blue').add_to(m)

    return m


In [ ]:
# Load the TSV file into a DataFrame
df = pd.read_csv('../data/places.tsv', sep='\t')
print(df.head())

In [35]:
# Compare results
results_df = compare_results(df)
print(results_df.head())

OpenAI result for Chavonis, fort: Name variants: Chavonis Fort, Chauonis Fort, Chawonis Fort

Language variants: English

Geographic coordinates: 
Latitude: 14.4000° N
Longitude: 75.5200° E
OpenAI result for Hutton In The Forest: Name variants: Hutton-in-the-Forest

Language variants: None

Geographic coordinates: 
Latitude: 54.7815° N
Longitude: 3.0718° W
OpenAI result for Pang: 1. Pang (English)
2. 庞 (Chinese)
3. 팡 (Korean)

Geographic coordinates:
Latitude: 37.8333° N
Longitude: 128.8833° E
OpenAI result for Warsop: Name variants:
1. Warsop (English)
2. Warsopu (Polish)

Language variants:
1. English
2. Polish

Geographic coordinates:
Latitude: 53.2136
Longitude: -1.1871
OpenAI result for Rothwell: Name variants: 
1. Rothwell (English)
2. Rodewelle (Old English)
3. Rodewall (Old English)
4. Rodewell (Old English)

Language variants:
1. Rothwell (English)
2. Rothwell (German)
3. Ротуэлл (Russian)
4. 罗思维尔 (Chinese)

Geographic coordinates:
Latitude: 53.7642° N
Longitude: 1.4876° W
Ope

In [36]:
# Generate map
map = generate_map(results_df)
map.save("map.html")

In [30]:
import openai
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
load_dotenv()

# Initialize OpenAI client
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
)

def getnames(place_name):
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"List all the name variants and language variants for the place, '{place_name}', and provide its geographic coordinates (latitude and longitude)."
            }
        ],
        model="gpt-3.5-turbo",
    )

    # Extracting the response content
    response_content = chat_completion.choices[0].message.content.strip()
    return response_content

if __name__ == "__main__":
    # Test with a single place name
    place_name = "Tavakli, Turkey"
    response_content = getnames(place_name)
    print(f"Raw Response for {place_name}:\n{response_content}")

Raw Response for Tavakli, Turkey:
Name variants: Tavaklı
Language variants: Tavaklı (Turkish)

Geographic coordinates:
Latitude: 40.369722
Longitude: 27.627778
